In [ ]:
!pip -q install sentence-transformers faiss-cpu groq \
langchain langchain-groq gspread

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3,
    max_retries=3
)

print("Groq LLM Connected Successfully!")

In [ ]:
BOOKS = {
    "atomic habits": {
        "title": "Atomic Habits",
        "author": "James Clear",
        "price": 450,
        "category": "non-fiction",
        "description": "A self-help book about building good habits "
                       "and breaking bad habits.",
        "stock": 20
    },

    "the alchemist": {
        "title": "The Alchemist",
        "author": "Paulo Coelho",
        "price": 350,
        "category": "fiction",
        "description": "An inspirational novel about following "
                       "your dreams and discovering your destiny.",
        "stock": 15
    },

    "harry potter": {
        "title": "Harry Potter",
        "author": "J.K. Rowling",
        "price": 550,
        "category": "children",
        "description": "A fantasy adventure about a young wizard "
                       "and his magical world.",
        "stock": 10
    },

    "rich dad poor dad": {
        "title": "Rich Dad Poor Dad",
        "author": "Robert Kiyosaki",
        "price": 400,
        "category": "non-fiction",
        "description": "A personal finance book about money, "
                       "investing and financial education.",
        "stock": 12
    },

    "wings of fire": {
        "title": "Wings of Fire",
        "author": "A.P.J. Abdul Kalam",
        "price": 300,
        "category": "non-fiction",
        "description": "An autobiography about the life, "
                       "education and achievements of A.P.J. Abdul Kalam.",
        "stock": 18
    }
}

SHIPPING = {
    "standard": 50,
    "express": 100
}

print("Total books:", len(BOOKS))

In [ ]:
chunks = []

for book_id, book in BOOKS.items():

    text = f"""
    Book Title: {book['title']}
    Author: {book['author']}
    Category: {book['category']}
    Price: Rs {book['price']}
    Description: {book['description']}
    """

    chunks.append(text.strip())

print("Total book chunks:", len(chunks))

for chunk in chunks:
    print(chunk)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

dimensions = embeddings.shape[1]

index = faiss.IndexFlatIP(dimensions)

index.add(embeddings)

print("Vector Dimensions:", dimensions)
print("Books Stored:", index.ntotal)

In [ ]:
def search_books(question, top_k=3):

    question_vector = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_vector = np.asarray(
        question_vector,
        dtype="float32"
    )

    scores, indices = index.search(
        question_vector,
        min(top_k, len(chunks))
    )

    results = []

    for score, i in zip(scores[0], indices[0]):

        if i >= 0:
            results.append({
                "content": chunks[i],
                "similarity": float(score)
            })

    return results

In [ ]:
# results = search_books(
#     "I want to learn about money management"
# )

# for result in results:

#     print("Similarity:", result["similarity"])
#     print(result["content"])
#     print("-" * 50)

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()

gc = gspread.authorize(creds)

sh = gc.create("ReadMore Book Orders")

ws = sh.sheet1

ws.append_row([
    "Name",
    "Book",
    "Quantity",
    "Category",
    "Shipping",
    "Total"
])

print("Bookstore Sheet Created!")
print(sh.url)

In [ ]:
from langchain.tools import tool
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
@tool
def get_book_price(book: str, qty: int = 1) -> str:
    """Check book price and calculate the total book cost."""

    book = book.lower().strip()

    if book not in BOOKS:
        return "Book not found in our bookstore."

    price = BOOKS[book]["price"]

    return (
        f"{qty} x {BOOKS[book]['title']} "
        f"= Rs {price * qty}"
    )

In [ ]:
@tool
def check_stock(book: str, qty: int = 1) -> str:
    """Check the available stock for a particular book."""

    book = book.lower().strip()

    if book not in BOOKS:
        return "Book not found."

    available = BOOKS[book]["stock"]

    if available >= qty:
        return (
            f"{BOOKS[book]['title']} is available. "
            f"Remaining stock: {available}"
        )

    return (
        f"Only {available} copies are available. "
        f"Requested quantity: {qty}"
    )

In [ ]:
@tool
def get_shipping_price(shipping: str) -> str:
    """Check standard or express shipping charges."""

    shipping = shipping.lower().strip()

    if shipping not in SHIPPING:
        return "Choose standard or express shipping."

    return (
        f"{shipping.title()} shipping costs "
        f"Rs {SHIPPING[shipping]}."
    )

In [ ]:
@tool
def recommend_books(interests: str) -> str:
    """Recommend books based on customer interests."""

    results = search_books(interests, top_k=3)

    if not results:
        return "No matching books found."

    recommendations = []

    for result in results:
        recommendations.append(
            result["content"]
        )

    return "\n\n".join(recommendations)

In [ ]:
class BookOrder(BaseModel):

    name: str = Field(
        description="Customer name"
    )

    book: Literal[
        "atomic habits",
        "the alchemist",
        "harry potter",
        "rich dad poor dad",
        "wings of fire"
    ]

    qty: int = Field(
        ge=1,
        description="Number of copies"
    )

    category: Literal[
        "fiction",
        "non-fiction",
        "academic",
        "children"
    ]

    shipping: Literal[
        "standard",
        "express"
    ]

In [ ]:
@tool
def log_order(
    name: str,
    book: str,
    qty: int,
    category: str,
    shipping: str
) -> str:
    """Save an order after customer confirmation and stock validation."""

    try:

        order = BookOrder(
            name=name,
            book=book.lower().strip(),
            qty=qty,
            category=category.lower().strip(),
            shipping=shipping.lower().strip()
        )

        available = BOOKS[order.book]["stock"]

        if available < order.qty:
            return (
                f"Order not saved. Only {available} "
                "copies are available."
            )

        book_total = (
            BOOKS[order.book]["price"] * order.qty
        )

        shipping_total = SHIPPING[order.shipping]

        total = book_total + shipping_total

        ws.append_row([
            order.name,
            order.book,
            order.qty,
            order.category,
            order.shipping,
            total
        ])

        BOOKS[order.book]["stock"] -= order.qty

        return (
            f"Order successfully saved! "
            f"{order.qty} x {BOOKS[order.book]['title']}. "
            f"Customer: {order.name}. "
            f"Shipping: {order.shipping}. "
            f"Total: Rs {total}."
        )

    except Exception as e:

        return f"Order could not be saved: {str(e)}"

In [ ]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly AI shopping assistant
for ReadMore Online Bookstore.

BOOKSTORE SERVICES:

We sell fiction, non-fiction, academic
and children's books.

Customers can search for books based on
their interests and preferences.

TOOLS:

1. recommend_books:
Use semantic search to recommend books
based on customer interests.

2. get_book_price:
Always use this tool to check book prices.

3. check_stock:
Always check stock before accepting an order.

4. get_shipping_price:
Use this tool to check shipping charges.

5. log_order:
Save confirmed customer orders to Google Sheets.

RULES:

- Never invent book information, prices or stock.
- Recommend books based on customer interests.
- Ask questions if book details are unclear.
- Collect customer name, book, quantity,
  category and shipping method.
- Check stock before confirming an order.
- Calculate the total using actual book and
  shipping prices.
- Show the complete order summary.
- Ask for explicit customer confirmation
  before calling log_order.
- Never save an order without confirmation.
- Never guarantee delivery dates.
- If a book is unavailable, suggest alternatives.

LANGUAGE:

Reply in friendly English or Telugu,
matching the customer's language.

Keep responses clear and concise.
"""

agent = create_agent(
    model=llm,
    tools=[
        recommend_books,
        get_book_price,
        check_stock,
        get_shipping_price,
        log_order
    ],
    system_prompt=SYSTEM
)

print("ReadMore AI Agent Ready!")

In [ ]:
history = []

def chat(text: str) -> str:

    global history

    try:

        result = agent.invoke({
            "messages": history + [
                {
                    "role": "user",
                    "content": text
                }
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:

        print("Agent Error:", e)

        return (
            "Sorry, I'm a little busy. "
            "Please try again."
        )

In [ ]:
print(chat(
    "Hi! I am looking for motivational books."
))

print(chat(
    "Suggest a book that helps me build good habits."
))

print(chat(
    "What is the price of Atomic Habits?"
))

print(chat(
    "Is Atomic Habits available?"
))

print(chat(
    "I am Ravi. I want 2 copies of Atomic Habits."
))

print(chat(
    "I want standard shipping."
))

print(chat(
    "The category is non-fiction."
))

print(chat(
    "Yes, please confirm my order."
))

In [ ]:
print("Welcome to ReadMore Online Bookstore!")
print("Type 'exit' to end the conversation.")

while True:

    question = input("\nYou: ").strip()

    if question.lower() == "exit":

        print("Thank you for visiting ReadMore!")
        break

    if not question:
        continue

    answer = chat(question)

    print("\nReadMore Assistant:", answer)